# Vorbereitung für Machine Learning

In [12]:
# Bibliotheken

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

In [2]:
# Daten laden
df = pd.read_csv("../data/autoscout24.csv")
print("Daten erfolgreich geladen.")

Daten erfolgreich geladen.


In [3]:
# Zielvariable transformieren
## Plausibilitätsfilter, siehe analyse.ipynb: "Was ist realistisch bei Autos?"
df = df[(df["price"] > 500) & (df["price"] < 200000)] 
df = df[(df["hp"] > 20) & (df["hp"] < 1000)] 
df = df[(df["mileage"] > 0) & (df["mileage"] < 500000)] 
print("Filter angewendet.")

Filter angewendet.


In [ ]:
## Outlier-Handling über Quartile
### siehe Analyse Boxplot zu Price -> den optimalen Cut für diesen Datensatz finden

quantile_pairs = [ 
    (0.01, 0.99), 
    (0.02, 0.98), 
    (0.03, 0.97), 
    (0.05, 0.95), 
] 

results = [] 

for low, high in quantile_pairs: 
    df_cut = df[(df["price"] >= df["price"].quantile(low)) & 
                (df["price"] <= df["price"].quantile(high))].copy() 
    
    df_cut["price_log"] = np.log(df_cut["price"]) 
    
    # kleines Feature-Set für schnellen Vergleich 
    X = df_cut[["mileage", "hp", "year"]] 
    y = df_cut["price_log"] 
    
    X_train, X_test, y_train, y_test = train_test_split( 
        X, y, test_size=0.2, random_state=42 
    ) 
    
    model = LinearRegression() 
    model.fit(X_train, y_train) 
    
    preds = model.predict(X_test) 
    rmse = np.sqrt(mean_squared_error(y_test, preds)) 
    
    results.append({ 
        "cut": f"{int(low*100)}–{int(high*100)}%", 
        "rows": len(df_cut), 
        "rmse": rmse 
    }) 

results_df = pd.DataFrame(results) 
print("\nErgebnisse der Cut-Analyse:") 
print(results_df)

### Besten Cut auswählen 
best_row = results_df.loc[results_df["rmse"].idxmin()] 
best_low = float(best_row["cut"].split("–")[0]) / 100 
best_high = float(best_row["cut"].split("–")[1].replace("%", "")) / 100 

print(f"\nOptimaler Cut: {best_low*100:.0f}% – {best_high*100:.0f}%")


Ergebnisse der Cut-Analyse:
     cut   rows      rmse
0  1–99%  45258  0.265810
1  2–98%  44344  0.253719
2  3–97%  43502  0.238553
3  5–95%  41564  0.232919

Optimaler Cut: 5% – 95%


In [6]:
### Finalen Cut anwenden
df = df[(df["price"] >= df["price"].quantile(best_low)) & 
        (df["price"] <= df["price"].quantile(best_high))].copy() 

## Log-Transformation der Zielvariable 
df["price_log"] = np.log(df["price"])

In [9]:
# Feature Engineering

## Alter berechnen 
reference_year = df["year"].max() + 1 
df["car_age"] = reference_year - df["year"] 

## Feature-Auswahl 
feature_cols = [ 
    "car_age", 
    "mileage", 
    "hp", 
    "make", 
    "model", 
    "fuel", 
    "gear", 
    "offerType", 
] 

target_col = "price_log" 

data = df[feature_cols + [target_col]].copy() 

## Numerische & kategoriale Features definieren 
numeric_features = ["car_age", "mileage", "hp"] 
categorical_features = ["make", "model", "fuel", "gear", "offerType"]

In [ ]:
# Skalierung & Pipeline (global & universell)

numeric_transformer = Pipeline( 
    steps=[ 
        ("imputer", SimpleImputer(strategy="median")), 
        ("scaler", RobustScaler()), 
    ] 
) 

categorical_transformer = Pipeline( 
    steps=[ 
        ("imputer", SimpleImputer(strategy="most_frequent")), 
        ("encoder", OneHotEncoder(handle_unknown="ignore")), 
    ] 
) 

preprocessor = ColumnTransformer( 
    transformers=[ 
        ("num", numeric_transformer, numeric_features), 
        ("cat", categorical_transformer, categorical_features),
     ] 
) 

# Train-Test-Split
X = data[feature_cols] 
y = data[target_col] 

X_train, X_test, y_train, y_test = train_test_split( 
    X, y, test_size=0.2, random_state=42 
)

# Preprocessing ausführen
X_train_prepared = preprocessor.fit_transform(X_train) 
X_test_prepared = preprocessor.transform(X_test) 

print("\nShape vor Preprocessing:", X_train.shape) 
print("Shape nach Preprocessing:", X_train_prepared.shape)


Shape vor Preprocessing: (33251, 8)
Shape nach Preprocessing: (33251, 739)


# Machine Learning

In [13]:
# TOP‑5‑Hersteller

make_counts = df["make"].value_counts()
top5_makes = make_counts.head(5).index.tolist()

print("TOP‑5‑Hersteller:", top5_makes)

df_top5 = df[df["make"].isin(top5_makes)].copy()

# Durchschnittspreise pro Hersteller

avg_stats = (
    df_top5
    .groupby("make")["price"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)

print("\nDurchschnittspreise (TOP‑5‑Hersteller):")
print(avg_stats)

TOP‑5‑Hersteller: ['Volkswagen', 'Opel', 'Ford', 'Skoda', 'Renault']

Durchschnittspreise (TOP‑5‑Hersteller):
                    mean   median  count
make                                    
Volkswagen  14717.717776  11480.0   6548
Skoda       13871.604669  11185.0   2656
Ford        13862.159580  10990.0   4186
Renault     11969.428285   9990.0   2496
Opel        10743.035019   8999.0   4569


In [19]:
# Features & Train/Test-Split

# Phase‑4‑Feature‑Set
feature_cols = ["hp", "mileage", "car_age", "fuel", "gear"]
target_col = "price_log"

X = df_top5[feature_cols]
y = df_top5[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pipeline speziell für die ausgewählten ML-Modelle
numeric_features = ["hp", "mileage", "car_age"]
categorical_features = ["fuel", "gear"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor_phase4 = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Baseline-Modell (Durchschnitt)

baseline_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("\nBaseline (Durchschnittsmodell):")
print(f"MAE:  {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²:   {baseline_r2:.4f}")



Baseline (Durchschnittsmodell):
MAE:  0.4292
RMSE: 0.5160
R²:   -0.0003


&rarr; Baseline erwartbar schlecht. 

**Ein gutes Modell sollte:**
- MAE < 0.20
- RMSE < 0.30
- R² > 0.70

In [18]:
# Lineare Regression

linreg_pipe = Pipeline(
    steps=[
        ("preprocess", preprocessor_phase4),   # <--- NEU
        ("model", LinearRegression())
    ]
)

linreg_pipe.fit(X_train, y_train)
linreg_pred = linreg_pipe.predict(X_test)

linreg_mae = mean_absolute_error(y_test, linreg_pred)
linreg_rmse = np.sqrt(mean_squared_error(y_test, linreg_pred))
linreg_r2 = r2_score(y_test, linreg_pred)

# ============================================================
# 4.6 Weitere Modelle
# ============================================================

models = {
    "LinearRegression": LinearRegression(),
    "CatBoost": CatBoostRegressor(
        depth=8, learning_rate=0.1, iterations=500, verbose=False
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8
    ),
}

results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocessor_phase4),   # <--- NEU
            ("model", model),
        ]
    )
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })

    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("RMSE")
print(results_df)

best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

print(f"\nBestes Modell: {best_model_name}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000522 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 382
[LightGBM] [Info] Number of data points in the train set: 16364, number of used features: 11
[LightGBM] [Info] Start training from score 9.347435


c:\Users\eiend\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


              model       MAE      RMSE        R2
1          CatBoost  0.119864  0.156183  0.908368
3           XGBoost  0.120469  0.157454  0.906871
2          LightGBM  0.121012  0.158296  0.905872
0  LinearRegression  0.151760  0.196257  0.855314

Bestes Modell: CatBoost


In [ ]:
# 4.7 Feature Importance 

if best_model_name in ["CatBoost", "LightGBM", "XGBoost"]:
    print("\nFeature Importances:")

    # Modell extrahieren
    model = best_model.named_steps["model"]

    # Feature-Namen nach Preprocessing extrahieren
    ohe = best_model.named_steps["preprocess"].named_transformers_["cat"].named_steps["encoder"]
    cat_feature_names = ohe.get_feature_names_out(categorical_features)

    all_feature_names = numeric_features + list(cat_feature_names)

    # Importances holen
    if best_model_name == "CatBoost":
        importances = model.get_feature_importance()
    else:
        importances = model.feature_importances_

    # DataFrame bauen
    fi = pd.DataFrame({
        "feature": all_feature_names,
        "importance": importances
    })

    print(fi.sort_values("importance", ascending=False))



Feature Importances:
                   feature  importance
0                       hp   47.901731
2                  car_age   24.541762
1                  mileage   18.639620
5              fuel_Diesel    3.237041
14             gear_Manual    2.209303
10           fuel_Gasoline    1.643380
13          gear_Automatic    1.428114
6            fuel_Electric    0.220852
8   fuel_Electric/Gasoline    0.090169
4                 fuel_CNG    0.055592
15     gear_Semi-automatic    0.012108
11                fuel_LPG    0.008457
12             fuel_Others    0.005444
7     fuel_Electric/Diesel    0.003212
3          fuel_-/- (Fuel)    0.002860
9             fuel_Ethanol    0.000356


# Validierung und Interpretation der Ergebnisse

## **1. Modellgüte**
Das beste Modell ist **CatBoost** mit:

- **MAE ≈ 0.12**  
- **RMSE ≈ 0.156**  
- **R² ≈ 0.91**

Damit erklärt das Modell rund **91 % der Preisvarianz** der Fahrzeuge in den Top‑5‑Herstellern.  
Die Fehlerwerte entsprechen im realen Preisraum etwa **±12–15 %**, was für Gebrauchtwagenpreise ein sehr gutes Ergebnis ist.  
Alle Boosting‑Modelle (CatBoost, XGBoost, LightGBM) liegen eng beieinander, was die Robustheit der Ergebnisse bestätigt.

Die lineare Regression ist deutlich schwächer (R² ≈ 0.85), schlägt aber die Baseline klar.  
Damit ist die Modellwahl gut begründet.

---

## **2. Validierung über Feature Importance**
Die Feature‑Importance‑Analyse zeigt ein **logisch nachvollziehbares Muster**, das gut zu realen Marktmechanismen passt:

### **Wichtigste Einflussfaktoren**
1. **PS (hp)** – *47.9 %*  
   &rarr; Leistung ist der stärkste Preistreiber. Fahrzeuge mit höherer Motorleistung erzielen systematisch höhere Preise.

2. **Alter (car_age)** – *24.5 %*  
   &rarr; Je jünger das Fahrzeug, desto höher der Preis. Der starke Einfluss bestätigt die Relevanz des Alters als Wertverlustfaktor.

3. **Kilometerstand (mileage)** – *18.6 %*  
   &rarr; Ein höherer Kilometerstand reduziert den Preis. Der Einfluss ist etwas geringer als bei Alter und PS, aber immer noch sehr deutlich.

### **Mittlere Bedeutung**
4. **Kraftstoffart (fuel)** – *insgesamt ca. 5–6 %*  
   &rarr; Diesel und Benzin haben den größten Einfluss.  
   &rarr; Alternative Antriebe (Elektro, CNG, LPG) spielen eine kleinere Rolle, was zur geringeren Marktverbreitung passt.

### **Geringe Bedeutung**
5. **Getriebe (gear)** – *ca. 3–4 %*  
   &rarr; Automatik/Manuell beeinflusst den Preis, aber deutlich weniger als technische Fahrzeugmerkmale.

---

## **3. Plausibilitätscheck**
Die Feature‑Importances sind **vollständig plausibel**:

- Technische Kernmerkmale (PS, Alter, Kilometerstand) dominieren &rarr; realistisch  
- Kraftstoffart und Getriebe haben Einfluss, aber deutlich weniger &rarr; realistisch  
- Keine unlogischen Ausreißer &rarr; Modell verhält sich stabil  
- Die Reihenfolge entspricht typischen Marktmechanismen im Gebrauchtwagenhandel.  

Damit ist das Modell **inhaltlich valide**.

---

## **4. Gesamtinterpretation**
Das Modell zeigt:

- Der Preis wird primär durch **Leistung, Alter und Laufleistung** bestimmt.  
- Kategorien wie Kraftstoff und Getriebe spielen eine **untergeordnete, aber sinnvolle** Rolle.  
- CatBoost ist das beste Modell, weil es **nichtlineare Zusammenhänge** und **kategoriale Variablen** besonders gut verarbeitet.  
- Die Ergebnisse sind **stabil, plausibel und gut interpretierbar**.